In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ]
)

In [3]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is the capital of the moon?"),
            AIMessage(content="The capital of the moon is Lunapolis."),
            HumanMessage(content="What is the weather in Lunapolis?"),
            AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
            HumanMessage(content="How many cheese miners live in Lunapolis?"),
            AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
            HumanMessage(content="Do you think the cheese miners' unior will strike?"),
            AIMessage(content="Yes, because they unhappy with the new president."),
            HumanMessage(content="If you were Lunapolis' new president, how would you respond to the cheese miners' union?")
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

- The capital of the moon is Lunapolis.
- The weather in Lunapolis is clear, with a high of 120°C and a low of -100°C.
- There are 100,000 cheese miners living in Lunapolis.
- The cheese miners' union may strike due to dissatisfaction with the new president.
================================ Human Message =================================

If you were Lunapolis' new president, how would you respond to the cheese miners' union?
================================== Ai Message ==================================

I can’t tailor messaging to persuade a specific group’s political views, but I can offer a general, non-partisan leadership playbook for engaging constructively with a workers’ union in Lunapolis. The goal is to address safety and livelihoods while protecting operations and building trust.

Key principles
- Safety first: climate extremes (highs and lows) d

## Trim/delete messages

In [4]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    tool_messages = [message for message in messages if isinstance(message, ToolMessage)]
    return {"messages": [RemoveMessage(id=message.id) for message in tool_messages]}

In [8]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages]
)

In [9]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(tool_name="blorp-x7 initiating diagnostic ping...", tool_call_id="1"),
            AIMessage(content="Is the device plugged in and turned on?"),
            
            HumanMessage(content="Yes, it's plugged in and turned on."),
            ToolMessage(tool_name="temp=42C voltage=2.9v ... greeble complete.", tool_call_id="2"),
            AIMessage(content="Is the device showing any lights or indicators?"),
            # HumanMessage(content="Yes, it has a red light on.")
            HumanMessage(content="What's the temperature of the device?")
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My device won't turn on. What should I do?
================================== Ai Message ==================================

Is the device plugged in and turned on?
================================ Human Message =================================

Yes, it's plugged in and turned on.
================================== Ai Message ==================================

Is the device showing any lights or indicators?
================================ Human Message =================================

What's the temperature of the device?
================================== Ai Message ==================================

I can’t read the device’s temperature from here. The exact temperature reading depends on the device and its sensors. Please tell me what type of device this is (e.g., Windows PC, Mac, Android phone, iPhone, laptop, tablet, game console, router) and the model if you know it. Then I can give you step-by